# Model Evaluation — Austrian Tax Law Q&A

Evaluate 3 models against ground-truth answers from the Austrian Tax Law Dataset using:
- **Exact Match** — strict string equality
- **BLEU** — n-gram precision
- **ROUGE** (1, 2, L) — recall-oriented n-gram overlap
- **BERTScore** — semantic similarity via contextual embeddings
- **Reference-free quality metrics** — answer length, repetition rate
- **Error analysis** — qualitative failure mode inspection

In [ ]:
!pip install rouge-score nltk bert-score pandas numpy matplotlib seaborn --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Load ground truth
gt = pd.read_csv('Austrian Tax Law Dataset - Dataset.csv')
gt = gt[['id', 'correct_answer']].copy()
gt['correct_answer'] = gt['correct_answer'].fillna('')

# Load model results
m1 = pd.read_csv('../results/model1_results.csv')
m2 = pd.read_csv('../results/model2_results.csv')
m3 = pd.read_csv('../results/model3_results.csv')

for df in [m1, m2, m3]:
    df['answer'] = df['answer'].fillna('')

# Merge each model with ground truth on id
m1_eval = m1.merge(gt, on='id', how='inner')
m2_eval = m2.merge(gt, on='id', how='inner')
m3_eval = m3.merge(gt, on='id', how='inner')

print(f'Ground truth: {len(gt)} answers')
print(f'Model 1 (Baseline GPT-2): {len(m1_eval)} matched')
print(f'Model 2 (Fine-tuned GPT-2): {len(m2_eval)} matched')
print(f'Model 3 (RAG + GPT-4o-mini): {len(m3_eval)} matched')

## 1. Reference-Free Metrics

Basic quality indicators that measure answer structure without comparing to ground truth.

In [ ]:
def compute_reference_free_metrics(answers):
    """Compute reference-free quality metrics for a list of answers."""
    lengths = [len(a) for a in answers]
    word_counts = [len(a.split()) for a in answers]
    
    # Repetition rate: ratio of unique trigrams to total trigrams
    rep_rates = []
    for a in answers:
        words = a.split()
        if len(words) < 3:
            rep_rates.append(0.0)
            continue
        trigrams = [tuple(words[i:i+3]) for i in range(len(words)-2)]
        rep_rates.append(len(set(trigrams)) / len(trigrams) if trigrams else 0.0)
    
    # Vocabulary diversity: unique words / total words
    all_words = ' '.join(answers).split()
    vocab_diversity = len(set(all_words)) / len(all_words) if all_words else 0.0
    
    empty_count = sum(1 for a in answers if len(a.strip()) <= 5)
    
    return {
        'Avg Chars': f'{np.mean(lengths):.0f}',
        'Avg Words': f'{np.mean(word_counts):.1f}',
        'Trigram Uniqueness': f'{np.mean(rep_rates):.3f}',
        'Vocab Diversity': f'{vocab_diversity:.3f}',
        'Empty/Trivial': f'{empty_count} ({empty_count/len(answers)*100:.1f}%)'
    }

models_raw = {'Model 1 (Baseline)': m1, 'Model 2 (Fine-tuned)': m2, 'Model 3 (RAG)': m3}
ref_free = {name: compute_reference_free_metrics(df['answer'].tolist()) for name, df in models_raw.items()}
ref_free['Ground Truth'] = compute_reference_free_metrics(gt['correct_answer'].tolist())

ref_free_df = pd.DataFrame(ref_free).T
ref_free_df

In [ ]:
# Visualize answer length distributions
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
all_data = {
    'Model 1 (Baseline)': m1['answer'],
    'Model 2 (Fine-tuned)': m2['answer'],
    'Model 3 (RAG)': m3['answer'],
    'Ground Truth': gt['correct_answer']
}
for i, (name, answers) in enumerate(all_data.items()):
    word_counts = answers.apply(lambda x: len(x.split()))
    axes[i].hist(word_counts, bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(name)
    axes[i].set_xlabel('Word count')
    axes[i].set_ylabel('Frequency')
    axes[i].axvline(word_counts.median(), color='red', linestyle='--', label=f'Median: {word_counts.median():.0f}')
    axes[i].legend()
plt.suptitle('Answer Length Distribution', fontsize=14)
plt.tight_layout()
plt.savefig('answer_lengths.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Exact Match

Strict comparison: does the prediction exactly match the reference (after normalization)?

In [ ]:
def normalize(text):
    return ' '.join(text.lower().split())

def exact_match_rate(predictions, references):
    matches = sum(1 for p, r in zip(predictions, references) if normalize(p) == normalize(r))
    return matches / len(predictions)

em_m1 = exact_match_rate(m1_eval['answer'].tolist(), m1_eval['correct_answer'].tolist())
em_m2 = exact_match_rate(m2_eval['answer'].tolist(), m2_eval['correct_answer'].tolist())
em_m3 = exact_match_rate(m3_eval['answer'].tolist(), m3_eval['correct_answer'].tolist())

print(f'Exact Match Rate (vs Ground Truth):')
print(f'  Model 1 (Baseline):   {em_m1:.4f}')
print(f'  Model 2 (Fine-tuned): {em_m2:.4f}')
print(f'  Model 3 (RAG):        {em_m3:.4f}')

## 3. BLEU Score

BLEU measures n-gram precision between generated answers and ground-truth references.
Higher = more n-gram overlap with the correct answer.

In [ ]:
import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def compute_bleu(predictions, references):
    """Compute average sentence-level BLEU-4 score with smoothing."""
    smooth = SmoothingFunction().method1
    scores = []
    for pred, ref in zip(predictions, references):
        ref_tokens = ref.split()
        pred_tokens = pred.split()
        if len(ref_tokens) == 0 or len(pred_tokens) == 0:
            scores.append(0.0)
            continue
        score = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth)
        scores.append(score)
    return np.mean(scores), scores

bleu1_avg, bleu1_scores = compute_bleu(m1_eval['answer'].tolist(), m1_eval['correct_answer'].tolist())
bleu2_avg, bleu2_scores = compute_bleu(m2_eval['answer'].tolist(), m2_eval['correct_answer'].tolist())
bleu3_avg, bleu3_scores = compute_bleu(m3_eval['answer'].tolist(), m3_eval['correct_answer'].tolist())

print(f'BLEU-4 Scores (vs Ground Truth):')
print(f'  Model 1 (Baseline):   {bleu1_avg:.4f}')
print(f'  Model 2 (Fine-tuned): {bleu2_avg:.4f}')
print(f'  Model 3 (RAG):        {bleu3_avg:.4f}')

## 4. ROUGE Scores

ROUGE measures recall of n-grams from the reference in the prediction.
- **ROUGE-1**: unigram overlap
- **ROUGE-2**: bigram overlap
- **ROUGE-L**: longest common subsequence

In [ ]:
from rouge_score import rouge_scorer

def compute_rouge(predictions, references):
    """Compute average ROUGE-1, ROUGE-2, ROUGE-L F1 scores."""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    r1, r2, rl = [], [], []
    for pred, ref in zip(predictions, references):
        if not pred.strip() or not ref.strip():
            r1.append(0.0); r2.append(0.0); rl.append(0.0)
            continue
        scores = scorer.score(ref, pred)
        r1.append(scores['rouge1'].fmeasure)
        r2.append(scores['rouge2'].fmeasure)
        rl.append(scores['rougeL'].fmeasure)
    return {'ROUGE-1': np.mean(r1), 'ROUGE-2': np.mean(r2), 'ROUGE-L': np.mean(rl)}

rouge_m1 = compute_rouge(m1_eval['answer'].tolist(), m1_eval['correct_answer'].tolist())
rouge_m2 = compute_rouge(m2_eval['answer'].tolist(), m2_eval['correct_answer'].tolist())
rouge_m3 = compute_rouge(m3_eval['answer'].tolist(), m3_eval['correct_answer'].tolist())

rouge_df = pd.DataFrame({
    'Model 1 (Baseline)': rouge_m1,
    'Model 2 (Fine-tuned)': rouge_m2,
    'Model 3 (RAG)': rouge_m3
}).T
rouge_df

## 5. BERTScore

BERTScore computes semantic similarity using contextual embeddings from a pre-trained BERT model.
Unlike BLEU/ROUGE, it captures meaning beyond surface-level n-gram overlap — important for
German legal text where correct answers can be paraphrased in many ways.

In [ ]:
from bert_score import score as bert_score

print('Computing BERTScore (this may take a few minutes)...')

# Use German language setting for multilingual BERT
P1, R1, F1_m1 = bert_score(m1_eval['answer'].tolist(), m1_eval['correct_answer'].tolist(), lang='de', verbose=False)
P2, R2, F1_m2 = bert_score(m2_eval['answer'].tolist(), m2_eval['correct_answer'].tolist(), lang='de', verbose=False)
P3, R3, F1_m3 = bert_score(m3_eval['answer'].tolist(), m3_eval['correct_answer'].tolist(), lang='de', verbose=False)

print(f'BERTScore F1 (vs Ground Truth):')
print(f'  Model 1 (Baseline):   P={P1.mean():.4f}  R={R1.mean():.4f}  F1={F1_m1.mean():.4f}')
print(f'  Model 2 (Fine-tuned): P={P2.mean():.4f}  R={R2.mean():.4f}  F1={F1_m2.mean():.4f}')
print(f'  Model 3 (RAG):        P={P3.mean():.4f}  R={R3.mean():.4f}  F1={F1_m3.mean():.4f}')

## 6. Main Results Table

In [ ]:
results = pd.DataFrame({
    'Model': ['Model 1 (Baseline GPT-2)', 'Model 2 (Fine-tuned GPT-2)', 'Model 3 (RAG + GPT-4o-mini)'],
    'Exact Match': [em_m1, em_m2, em_m3],
    'BLEU-4': [bleu1_avg, bleu2_avg, bleu3_avg],
    'ROUGE-1': [rouge_m1['ROUGE-1'], rouge_m2['ROUGE-1'], rouge_m3['ROUGE-1']],
    'ROUGE-2': [rouge_m1['ROUGE-2'], rouge_m2['ROUGE-2'], rouge_m3['ROUGE-2']],
    'ROUGE-L': [rouge_m1['ROUGE-L'], rouge_m2['ROUGE-L'], rouge_m3['ROUGE-L']],
    'BERTScore F1': [F1_m1.mean().item(), F1_m2.mean().item(), F1_m3.mean().item()]
})

# Format
display_results = results.copy()
for col in display_results.columns[1:]:
    display_results[col] = display_results[col].apply(lambda x: f'{x:.4f}')

display_results.set_index('Model')

In [ ]:
# Save results table
results.to_csv('evaluation_results.csv', index=False)
print('Saved evaluation_results.csv')

## 7. Metric Comparison Visualization

In [ ]:
metrics_to_plot = ['BLEU-4', 'ROUGE-1', 'ROUGE-L', 'BERTScore F1']
model_names = ['Model 1\n(Baseline)', 'Model 2\n(Fine-tuned)', 'Model 3\n(RAG)']

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_to_plot))
width = 0.25

for i, (_, row) in enumerate(results.iterrows()):
    vals = [row[m] for m in metrics_to_plot]
    ax.bar(x + i*width, vals, width, label=model_names[i])

ax.set_ylabel('Score')
ax.set_title('Model Comparison — Metrics vs Ground Truth')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot)
ax.legend()
ax.set_ylim(0, max(results[metrics_to_plot].max()) * 1.15)
plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Error Analysis

Qualitative inspection of failure modes across models.

In [ ]:
# Load questions for display
questions = pd.read_csv('../dataset_clean.csv')

# Pick 5 diverse questions
sample_ids = ['CORP-TAX-001', 'CORP-TAX-014', 'VAT-DOM-015', 'ESTG-NEW-010', 'SELF-014']

for qid in sample_ids:
    q_row = questions[questions['id'] == qid]
    gt_row = gt[gt['id'] == qid]
    if q_row.empty or gt_row.empty:
        continue
    question = q_row.iloc[0]['prompt']
    correct = gt_row.iloc[0]['correct_answer']
    a1 = m1[m1['id'] == qid].iloc[0]['answer'] if qid in m1['id'].values else 'N/A'
    a2 = m2[m2['id'] == qid].iloc[0]['answer'] if qid in m2['id'].values else 'N/A'
    a3 = m3[m3['id'] == qid].iloc[0]['answer'] if qid in m3['id'].values else 'N/A'
    
    print(f'=== {qid} ===')
    print(f'Q: {question}')
    print(f'CORRECT: {correct[:200]}')
    print(f'Model 1: {a1[:200]}')
    print(f'Model 2: {a2[:200]}')
    print(f'Model 3: {a3[:200]}')
    print()

In [ ]:
# Classify error types per model
def classify_errors(answers):
    """Classify common error types."""
    errors = {'trivial_short': 0, 'repetitive': 0, 'reasonable': 0}
    for a in answers:
        words = a.split()
        if len(words) <= 5:
            errors['trivial_short'] += 1
            continue
        trigrams = [tuple(words[i:i+3]) for i in range(len(words)-2)]
        uniqueness = len(set(trigrams)) / len(trigrams) if trigrams else 1.0
        if uniqueness < 0.3:
            errors['repetitive'] += 1
            continue
        errors['reasonable'] += 1
    return errors

error_types = {
    'Model 1 (Baseline)': classify_errors(m1['answer'].tolist()),
    'Model 2 (Fine-tuned)': classify_errors(m2['answer'].tolist()),
    'Model 3 (RAG)': classify_errors(m3['answer'].tolist())
}

error_df = pd.DataFrame(error_types).T
error_df['total'] = error_df.sum(axis=1)
print('Error classification (count):')
print(error_df)
print()

error_pct = error_df.div(error_df['total'], axis=0).drop(columns='total') * 100
print('Error classification (%):')
print(error_pct.round(1))

In [ ]:
# Per-question BERTScore distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (name, scores) in enumerate([('Model 1', F1_m1), ('Model 2', F1_m2), ('Model 3', F1_m3)]):
    axes[i].hist(scores.numpy(), bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{name} — BERTScore F1')
    axes[i].set_xlabel('BERTScore F1')
    axes[i].set_ylabel('Frequency')
    axes[i].axvline(scores.mean(), color='red', linestyle='--', label=f'Mean: {scores.mean():.3f}')
    axes[i].legend()
plt.suptitle('BERTScore F1 Distribution per Question', fontsize=14)
plt.tight_layout()
plt.savefig('bertscore_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Error type visualization
fig, ax = plt.subplots(figsize=(10, 5))
error_pct.plot(kind='bar', ax=ax, rot=0)
ax.set_ylabel('Percentage of answers (%)')
ax.set_title('Error Type Distribution by Model')
ax.legend(title='Error Type')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('Evaluation complete. Results and figures saved to evaluation/ folder.')